# 📗 정보 추출과 NER: 문서에서 개체를 뽑기

앞선 단원까지 우리는 약물·유전자·질병이 **노드**로, 그 사이의 사실이 **관계**로 들어간 지식 그래프를 갖췄습니다. 그 위에서 검색도 하고 질문에도 답해 봤습니다.

문제는 그 그래프가 **2016년에 정리된 데이터**라는 점입니다. 그 뒤에 나온 논문의 사실은 그 안에 없습니다. 그런데 새 논문은 매일 쏟아집니다. 사람이 읽고 손으로 노드와 관계를 넣는 방식으로는 따라갈 수 없습니다. **논문 원문에서 개체와 관계를 뽑아 그래프에 얹는 일**을 프로그램이 해야 합니다.

그 일이 이번 단원의 주제이고, 이름이 **정보 추출(IE, Information Extraction)** 입니다. 오늘은 그 첫 재료인 **개체**를 뽑습니다. 두 가지를 합니다.

- **뽑은 개체를 어떤 형식으로 적는가**: 원문에서의 **구간**과 **유형**, 그리고 학습형 모델이 쓰는 **BIO 태깅**입니다(2절).
- **개체를 실제로 어떻게 뽑는가**: 사람이 사전을 적어 두는 **규칙 기반**(3절)과 라벨 데이터로 학습된 **머신러닝 기반**(4절)을 순서대로 직접 돌려 봅니다.

같은 발췌에 두 방법을 써 보면 각자 **어디까지 잡고 어디서 막히는지**를 확인할 수 있습니다. 막히는 지점이 다음 시간에 **LLM** 을 쓰는 이유입니다.

## ⏪ 복습: 지난 시간까지

- **GraphRAG** 를 완성했습니다. 벡터 인덱스로 뜻이 가까운 논문을 찾고, PageRank·커뮤니티로 순위와 범위를 다듬은 뒤, 그 근거로 답을 만들었습니다.
- 그때 문서를 개체 노드에 이은 방법은 **이름 매칭**이었습니다. 본문에 사전의 이름이 그대로 나오면 `MENTIONS` 로 이었고, 거친 방법이라고 짚어 두었습니다. 오늘 그 방법을 직접 만들어 어디서 틀리는지 확인합니다.
- **지식 그래프**의 노드 레이블은 다섯이었습니다. `Compound`(약물)·`Gene`(유전자)·`Disease`(질병)·`Symptom`(증상)·`PharmacologicClass`(약효 분류). 오늘 뽑을 개체의 유형이 그대로 이 다섯입니다(**2-4** 에서 다시 씁니다).

**오늘의 목표**

**1. 정보 추출(IE)의 큰 그림**
- [ ] (1-1) **IE** 가 비정형 텍스트를 정형 지식으로 바꾸는 일임을 말한다.
- [ ] (1-2) IE 의 **네 단계**를 순서대로 말하고, NER 이 그중 첫 단계임을 말한다.

**2. 개체명 인식(NER)**
- [ ] (2-1) NER 의 출력인 **(구간, 유형)** 쌍을 `find` 로 직접 만들고 원문으로 되짚어 검증한다.
- [ ] (2-2) 구간을 **BIO 태깅**으로 적고, `B-` 와 `I-` 를 나누는 이유를 말한다.
- [ ] (2-3) 이 단원이 쓰는 **유형 5종**이 왜 그래프 노드 레이블과 같은 문자열인지 말한다.
- [ ] (2-4) NER 기법의 **계보**(규칙·CRF·BERT·LLM)를 정리한다.

**3. 규칙 기반 NER 체험**
- [ ] (3-1) 정규식 토크나이저와 **표준 사전**(`name2id.json`)으로 규칙 기반 NER 의 부품을 갖추고, 규칙 기반의 **두 축**(사전·패턴)을 가른다.
- [ ] (3-2~3-3) 사전과 낱말을 **모두 소문자로 바꿔** 맞출 때 생기는 **오탐**(`large`·상품명)을 눈으로 본다.
- [ ] (3-4) **대소문자를 지켜** 고치고 오탐이 사라지는 것을 확인한다.
- [ ] (3-5) 그래도 못 잡는 **네 가지 한계**(미등록어·경계·대소문자·약어 충돌)를 실행으로 확인한다.

**4. 머신러닝 기반 NER 체험**
- [ ] (4-1) **학습된 NER 모델**(`dslim/bert-base-NER`)을 돌려 BIO 라벨이 실제 출력 형식임을 확인하고, `aggregation_strategy` 로 **(구간, 유형)** 을 얻는다.
- [ ] (4-2) `pipeline` 의 **입력과 출력**이 무엇인지 칸 단위로 말한다.
- [ ] (4-3) 같은 모델을 우리 논문에 써서 **유형 목록이 학습할 때 고정된다**는 것을 확인한다.
- [ ] (4-4) 가중치 없이 **설정 파일만** 받아 모델이 내는 유형을 확인하는 법을 익힌다.

아래 준비 셀을 먼저 실행하세요. **본인 OpenAI API 키가 필요합니다**(`.env` 의 `OPENAI_API_KEY`). 이 시간의 코드가 OpenAI 모델을 부르지는 않지만, 준비 셀이 키를 먼저 확인하고 없으면 그 자리에서 멈춥니다.

> **4절에서 허깅페이스 NER 모델을 약 430MB 내려받습니다.** 인터넷 연결이 필요하고 처음 한 번만 걸립니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 데이터 파일 읽기: 이 셀은 실행만 하세요.
import json
from pathlib import Path

# 교안은 단원 폴더에서, 정답 노트북은 정답/ 폴더에서 돌아가므로 두 경로를 모두 본다.
_DATA = Path("data") if Path("data").exists() else Path("../data")


def load_jsonl(name):
    """data/<name> 을 한 줄씩 읽어 dict 리스트로 돌려준다(한 줄에 JSON 하나)."""
    rows = []
    for line in (_DATA / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_json(name):
    """data/<name> 을 통째로 읽어 dict 로 돌려준다(사전 파일용)."""
    return json.loads((_DATA / name).read_text(encoding="utf-8"))


print("데이터 폴더:", _DATA)

---
# 1. 정보 추출(IE)의 큰 그림

논문에서 그래프로 가려면 먼저 **무엇을 어떤 순서로 뽑을지**를 정해야 합니다. 이 대단원이 그 순서를 정리합니다.

- **1-1** IE 란 무엇인가
- **1-2** IE 의 네 단계

## 1-1. IE 란 무엇인가

### 왜 필요할까요?
논문·진료 기록·설명서는 사람이 읽으라고 쓴 **비정형 텍스트**입니다. 문장 안에 사실이 들어 있어도, 컴퓨터는 그것을 다른 사실과 잇거나 쿼리로 찾을 수 없습니다. 그러려면 문장을 먼저 **정형 지식**, 즉 **개체와 그 사이의 관계**로 바꿔야 합니다. 이 변환 작업이 **정보 추출(IE, Information Extraction)** 입니다.

### IE 의 입력과 출력
- 입력은 사람이 쓴 **자유로운 문장**입니다.
- 출력은 컴퓨터가 다룰 수 있는 **개체와 그 사이의 관계**입니다.
- 예: "Carbidopa 가 AHR 을 활성화한다고 보고됐다" → 개체 **Carbidopa**·**AHR** 을 찾고, 둘을 잇습니다.

IE 는 문서를 그래프로 바꾸는 작업이고, **네 단계**로 나뉩니다.

## 1-2. IE 의 네 단계

| 단계 | 하는 일 | 언제 |
|---|---|---|
| **개체 인식** | 문장에서 약물·유전자·질병 같은 **개체**를 잡기 | **오늘** |
| 스키마 설계 | 어떤 타입과 관계만 쓸지 **미리 정하기** | 이 단원 뒤에서 |
| 관계 추출 | 개체 **사이의 관계**를 뽑기 | 이 단원 뒤에서 |
| 정규화 | 같은 대상의 여러 표기를 **하나로** 통합 | 이 단원 뒤에서 |

<img src="images/ie_pipeline.png" width="760">

> 오늘은 네 단계 중 **첫 단계인 개체 인식**만 다룹니다. 나머지 세 단계는 이름만 기억해 두면 됩니다. 이 단원을 지나며 표에 적힌 순서대로 하나씩 배웁니다(**스키마 설계가 가장 먼저**입니다).
>
> 이 순서는 **개념의 순서**이지 반드시 네 번 나눠 호출하라는 뜻은 아닙니다. 실무에서는 LLM 한 번에 개체와 관계를 함께 받기도 합니다. 그래도 개념적으로는 개체가 먼저입니다. 관계를 걸 **양 끝**이 개체이기 때문입니다.

> **이 표는 우리 파이프라인의 네 단계입니다.** IE 라는 분야에는 이 넷 말고도 대표적인 과제가 둘 더 있습니다. 이름만 알아 두세요.
>
> - **이벤트 추출(event extraction)**: "누가·언제·무엇을 했는지"를 **한 덩어리 사건**으로 뽑는 일입니다. "2024년 3월, A사가 B사를 인수했다"에서 인수라는 사건 하나에 참여자와 시점을 함께 묶습니다.
> - **상호참조 해결(coreference resolution)**: 같은 문서 안의 `it`·`the drug`·`이 약물` 이 앞의 **어느 개체를 가리키는지** 잇는 일입니다. 이걸 안 하면 대명사로 받은 문장의 사실이 통째로 사라집니다.
>
> 우리 그래프에는 사건 노드가 없고 발췌도 짧아 이 단원에서는 다루지 않습니다. 다만 실무 문서에서 IE 를 하면 반드시 만납니다.

### ✅ 바로 확인 퀴즈

**1.** 정보 추출(IE)의 네 단계를 **순서대로** 바르게 나열한 것은?

<details><summary>정답 보기</summary>

**개체 인식 → 스키마 설계 → 관계 추출 → 정규화** 입니다. 먼저 개체를 잡고, 어떤 타입과 관계만 쓸지 계약으로 정한 뒤, 그 계약대로 관계를 뽑고, 마지막으로 같은 대상의 여러 표기를 하나로 통합합니다.

</details>

---
# 2. 개체명 인식(NER)

IE 네 단계 중 첫 단계인 **개체 인식**을 다룹니다. 여기서는 **표기와 유형**을 정합니다. 개체가 무엇인지 정의하고, NER 의 출력인 **(구간, 유형)** 쌍과 그것을 토큰마다의 라벨로 옮겨 적는 **BIO 태깅**을 직접 만듭니다. 실제로 개체를 뽑는 일은 **3절(규칙 기반)** 과 **4절(머신러닝 기반)** 에서 합니다.

- **2-1** NER 의 출력: (구간, 유형) 쌍
- **2-2** BIO 태깅: 학습형 NER 모델이 쓰는 라벨 형식
- **2-3** 이 단원이 쓰는 개체 유형 다섯 가지
- **2-4** 기법의 계보: 규칙에서 LLM 까지

## 2-1. NER 의 출력: (구간, 유형) 쌍

### 왜 필요할까요?
논문에서 지식을 뽑는 첫 걸음은 **이름을 찾는 일**입니다. "**Carbidopa** 는 **Parkinson disease** 치료에 쓰이며 **AHR** 을 활성화한다"에서 약물·질병·유전자에 해당하는 부분을 각각 찾아 표시해야 합니다.

### NER 의 정의
**개체명 인식(NER, Named Entity Recognition)** 은 **문장에서 개체가 있는 자리를 찾고, 그 개체의 유형을 붙이는 작업**입니다. 입력은 문장 하나이고, 출력은 찾은 개체마다 **(구간, 유형)** 쌍 하나입니다.

정의에 쓰인 낱말 셋의 뜻을 먼저 정합니다.

- **개체(entity, 엔티티)**: **우리가 미리 정해 둔 유형 목록에 드는 이름**입니다. 고유한 이름을 가진 대상만 개체인 것이 아닙니다. `statins` 처럼 약의 부류를 가리키는 말도 `PharmacologicClass` 라는 유형을 목록에 넣어 뒀으니 개체입니다.
- **유형(type)**: 그 개체가 무엇인지 가리키는 이름표입니다. 약물·유전자·질병 같은 것이고, 이 단원이 쓸 다섯 가지는 **2-3** 에서 정합니다.
- **구간(span)**: 그 개체가 원문의 **몇 번째 글자부터 몇 번째 글자까지**인지를 가리키는 숫자 두 개입니다.

### NER 의 출력
- 찾은 개체마다 **구간**과 **유형**을 짝지어 적습니다. 이 **(구간, 유형)** 쌍이 NER 이 내놓는 전부입니다.
- 뒤에서 관계를 뽑을 때는 이 개체들이 연결의 양 끝이 됩니다.

아래에서 그 쌍을 **직접 만들어** 봅니다. 논문 발췌에서 따온 문장 하나로 시작합니다.

<img src="images/ner_span_label.png" width="900">

> 문장 위에 칠한 자리가 **구간**, 그 위에 붙인 이름표가 **유형**입니다.

In [ ]:
# NER 의 출력은 (구간, 유형) 쌍이다. 구간은 원문에서 '몇 번째 글자부터 몇 번째 글자까지'인지를 가리킨다.
text = 'Carbidopa, used to treat Parkinson disease, was reported to activate AHR.'
labeled = [('Carbidopa', 'Compound'), ('Parkinson disease', 'Disease'), ('AHR', 'Gene')]   # 사람이 눈으로 찾은 개체

# 1) 이름마다 시작 위치를 찾아 (구간, 유형) 을 갖춘 dict 로 만든다
spans = []
for name, entity_type in labeled:
    start = text.find(name)                  # 이름이 시작하는 글자 번호. 원문에 없으면 -1 이 돌아온다
    spans.append({'name': name, 'type': entity_type,
                  'start': start, 'end': start + len(name)})   # end 는 '마지막 글자 다음'이라 슬라이싱에 그대로 쓴다

# 2) 만든 구간이 맞는지 원문을 되짚어 검증한다(개체가 3개라 3줄)
for span in spans:
    # 오른쪽이 개체 이름과 같아야 구간이 맞은 것이다
    print(span['start'], span['end'], span['type'], '->', text[span['start']:span['end']])

### 왜 이름이 아니라 구간인가

- 같은 이름이 한 논문에 **여러 번** 나오면 이름만으로는 어느 쪽을 가리키는지 정할 수 없습니다. 구간은 위치가 다르므로 둘을 구분합니다.
- 구간이 있으면 원문을 **되짚어** 검증할 수 있습니다. 위 출력에서 `text[start:end]` 가 개체 이름과 똑같이 찍히는 것이 그 검증입니다.
- 다만 **LLM 은 글자 번호를 잘 못 셉니다**. 그래서 다음 시간에는 구간 대신 **개체 텍스트**를 받고, 그 이름이 원문에 나오는지로 대신 검증합니다.

### 🖐️ 함께 따라하기: 사내 기술 문서에서 구간 만들기

이번엔 **의료 논문이 아니라 사내 기술 문서**로 같은 일을 해 봅니다. 문서가 달라지면 뽑을 유형도 달라집니다. 여기서는 `Framework`·`Tool` 두 가지를 씁니다(유형 목록을 쓸 데에 맞춰 정하는 이야기는 **2-3** 에서 합니다). 이 과정의 단위 프로젝트도 사내 기술 문서에서 개체를 뽑는 일이라, 같은 도메인을 미리 한 번 만져 보는 셈입니다.

아래 문장을 `sentence` 에 담고, 개체 세 개의 구간을 `find` 로 구해 출력해 보세요.

```text
결제 서비스는 Spring Boot 로 API 를 제공하고, 이벤트 로그는 Kafka 로 모아 Elasticsearch 에 적재합니다.
```

- 개체는 `('Spring Boot', 'Framework')`·`('Kafka', 'Tool')`·`('Elasticsearch', 'Tool')` 입니다.
- **확인 기준**: 세 개체 모두 `start` 가 `-1` 이 아니고, `sentence[start:end]` 가 개체 이름과 **똑같이** 찍히면 맞은 것입니다. `Spring Boot` 처럼 **공백이 든 이름**도 `find` 는 통째로 찾습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 위 문장을 sentence 변수에 담는다
# 2) [('Spring Boot', 'Framework'), ('Kafka', 'Tool'), ('Elasticsearch', 'Tool')] 를 돌며 find 로 start 를 구한다
# 3) {'name','type','start','end'} dict 를 만들어 sentence[start:end] 와 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** NER 의 출력을 가장 정확히 설명한 것은? (문장의 요약문 / 문장과 질문의 유사도 점수 / 개체의 구간과 유형 / 문장의 감정 라벨)

<details><summary>정답 보기</summary>

문장 속 **개체의 구간(span)과 그 유형**입니다. 문장을 요약하거나 유사도를 재는 것이 아니라, "어디에 어떤 유형의 개체가 있는지"를 찾습니다.

</details>

## 2-2. BIO 태깅: 학습형 NER 모델이 쓰는 라벨 형식

### 왜 필요할까요?
2-1 에서 만든 것은 **개체 세 개짜리 목록**이었습니다. 그런데 NER 모델은 문장을 **토큰 하나씩** 읽으면서 토큰마다 답을 하나씩 내놓습니다. 그래서 모델을 학습시키려면 정답도 같은 모양이어야 합니다. **토큰마다 라벨 하나**입니다.

여기서 문제가 하나 생깁니다. `Parkinson disease` 는 개체 **하나**인데 토큰은 **둘**입니다. 두 토큰에 그냥 `Disease` 라고만 적어 두면, 나중에 그 라벨만 보고 **질병 하나가 두 토큰에 걸친 것인지, 질병 둘이 나란히 있는 것인지** 알 수 없습니다.

**BIO 태깅**은 라벨 앞에 글자 하나를 붙여 이 문제를 풉니다.

### 문법: B·I·O 세 라벨

세 글자는 각각 영어 낱말의 첫 글자입니다.

| 라벨 | 무엇의 약자 | 언제 붙이나 |
|---|---|---|
| `B-유형` | **B**egin (시작) | 개체가 **시작**하는 토큰 |
| `I-유형` | **I**nside (안쪽) | 그 개체가 **이어지는** 토큰 |
| `O` | **O**utside (바깥) | 개체가 아닌 토큰. 유형을 안 붙입니다 |

### 예시: 2-1 의 그 문장에 직접 붙여 보면

개체는 `Carbidopa`(Compound)·`Parkinson disease`(Disease)·`AHR`(Gene) 셋이었습니다. 공백으로 끊은 토큰마다 라벨을 하나씩 붙이면 이렇게 됩니다.

| 토큰 | 라벨 | 왜 이 라벨인가 |
|---|---|---|
| `Carbidopa,` | `B-Compound` | 약물 개체가 **여기서 시작**한다 |
| `used` | `O` | 개체가 아니다 |
| `to` | `O` | |
| `treat` | `O` | |
| `Parkinson` | `B-Disease` | 질병 개체가 **여기서 시작**한다 |
| `disease,` | `I-Disease` | 앞 개체가 **이어진다**. 두 토큰이 한 덩어리 |
| `was` | `O` | |
| `reported` | `O` | |
| `to` | `O` | |
| `activate` | `O` | |
| `AHR.` | `B-Gene` | 유전자 개체가 **여기서 시작**한다 |

규칙은 세 줄이 전부입니다. **개체가 시작하는 자리에만 `B-`**, 그 개체가 **이어지는 동안 `I-`**, 나머지는 전부 `O`. 개체가 셋이라 `B-` 가 셋이고, `Parkinson disease` 만 두 토큰이라 `I-` 가 하나 붙었습니다.

> **모델이 고르는 선택지는 몇 개일까요.** 유형이 다섯이면 `B-` 다섯 + `I-` 다섯 + `O` 하나 = **11개**입니다. **4절**에서 돌려 볼 `dslim/bert-base-NER` 은 유형이 넷이라 **9개**입니다. 모델은 토큰마다 이 중 하나를 고르는 것이고, 그래서 NER 을 **토큰 분류(token classification)** 라고 부릅니다.

### 왜 `B-` 와 `I-` 를 굳이 나누나

`B-` 없이 유형만 적어도 될 것 같습니다. 그런데 **같은 유형의 개체 둘이 나란히** 오면 구분이 안 됩니다. 유전자 기호 두 개가 붙어 있는 자리를 봅시다.

| 토큰 | 태그 A | 태그 B |
|---|---|---|
| `CYP2C19` | `B-Gene` | `B-Gene` |
| `SLCO1B1` | `B-Gene` | `I-Gene` |

- **A** 는 `B-` 가 두 번이니 유전자가 **두 개**입니다.
- **B** 는 `I-` 로 이어졌으니 이름이 두 낱말인 유전자 **한 개**입니다.

토큰은 똑같은데 태그 한 글자가 뜻을 갈랐습니다. 만약 둘 다 `Gene Gene` 이라고만 적었다면 이 둘을 구분할 방법이 없습니다. **`B-` 는 바로 이 자리를 위해 있습니다.**

<img src="images/bio_boundary.png" width="900">

### `I-` 는 공백이 아니라 토큰을 따라갑니다

위 예시가 공백으로 갈린 낱말들이라 "**낱말 사이에 공백이 있으면 `I-`**" 로 읽기 쉽습니다. 그렇지 않습니다. `I-` 가 붙는 기준은 **한 개체가 토큰 몇 개에 걸쳐 있느냐**이고, 토큰을 어디서 끊을지는 **토크나이저가 정합니다.**

우리 데모는 `split()` 으로 공백에서 끊으니 `Parkinson disease` 가 두 토큰입니다. 그런데 **4절**에서 돌려 볼 학습된 모델은 낱말을 더 잘게 쪼갭니다. 공백이 하나도 없는 `Neo4j` 한 낱말이 세 토큰이 되고, 라벨은 이렇게 붙습니다(`ORG` 는 그 모델이 쓰는 **기관** 유형입니다. 그 모델의 유형 목록은 **4-1** 에서 정리합니다).

| 토큰 | `Neo` | `##4` | `##j` |
|---|---|---|---|
| 라벨 | `B-ORG` | `I-ORG` | `I-ORG` |

공백이 없는데도 `I-` 가 두 번 붙었습니다(`##` 는 앞 토큰에 붙는다는 표시입니다). **`I-` 는 "앞에서 시작한 그 개체가 아직 안 끝났다"는 뜻**이지, 공백에 대한 표시가 아닙니다.

### 같은 이름이 또 나오면 다시 `B-`

BIO 는 개체가 **몇 종류 있는지**가 아니라 **어느 자리에 있는지**를 적는 표기입니다. 그래서 같은 이름이 뒤에 또 나오면 그 자리에서 **다시 `B-` 로 시작**합니다. 앞에 나온 것과 같은 대상이어도 마찬가지입니다.

> `Daniel Park joined Elastic. Later Daniel Park left Elastic for Amsterdam.`

이 문장에서 `Daniel Park` 와 `Elastic` 은 각각 **두 번** 나옵니다. 학습된 모델에 넣으면 네 자리 **모두에 `B-` 가 새로 붙고**, 합친 결과도 (`Amsterdam` 까지) **다섯 개**가 됩니다. 즉 **`B-` 의 개수는 개체가 등장한 횟수**이지 서로 다른 개체의 수가 아닙니다.

> 같은 대상을 가리키는 여러 자리를 **하나로 묶는 일**은 BIO 가 하지 않습니다. IE 네 단계의 마지막인 **정규화**가 하는 일이고, 이 단원 뒤에서 다룹니다.

아래 데모는 **첫 등장만** 태깅합니다. 같은 이름을 다시 만나면 멈추는 수업용 단순화라, 코드 안에 그렇게 적어 두었습니다. 실제 태깅기는 모든 등장 자리를 찾습니다.

이제 맨 위 예시 표를 코드로 만들어 봅니다. 사람이 표시한 `(이름, 유형)` 에서 출발해 토큰마다의 라벨을 채웁니다.

In [ ]:
# BIO 태깅: 구간을 '토큰마다의 라벨'로 바꿔 적는다. NER 모델이 학습하는 형식이 이것이다.
tokens = text.split()                        # 영어는 공백으로 끊으면 낱말이 된다. 구두점은 낱말에 붙은 채 남는다('AHR.')

tags = ['O'] * len(tokens)                   # 먼저 전부 O 로 두고, 개체에 걸린 토큰만 덮어쓴다
for name, entity_type in labeled:
    parts = name.split()                     # 'Parkinson disease' 는 토큰 두 개에 걸친 개체다
    for i, token in enumerate(tokens):
        if parts[0] in token:                # 쉼표나 마침표가 붙으므로 완전일치가 아니라 '포함'으로 찾는다
            for j, part in enumerate(parts):
                tags[i + j] = ('B-' if j == 0 else 'I-') + entity_type   # 첫 토큰만 B, 이어지는 토큰은 I
            break                            # 같은 이름이 뒤에 또 나와도 첫 자리만 태깅하는 수업용 단순화다

# 토큰 하나에 라벨 하나가 짝지어졌는지 눈으로 확인한다. 이 짝이 곧 BIO 표기다.
for token, tag in zip(tokens, tags):
    print(token, '->', tag)

> 토큰마다 라벨이 하나씩 붙었습니다. 다만 `Carbidopa,`·`AHR.` 처럼 **구두점이 토큰에 붙어** 있습니다. `split()` 으로 공백만 보고 끊었기 때문입니다. 그래서 실무 NER 은 BIO 태그와 함께 **글자 번호(`start`·`end`)** 를 같이 들고 다닙니다. 토큰으로 복원한 이름은 원문과 글자가 어긋날 수 있지만, 글자 번호는 원문을 그대로 가리킵니다.
>
> BIO 는 **토큰 하나에 라벨 하나**라, `proton pump inhibitors (PPIs)` 처럼 개체가 **겹치는** 자리는 적을 수 없습니다. 그럴 때는 **2-1 의 구간 목록**을 그대로 들고 다닙니다. 구간은 서로 겹쳐도 되기 때문입니다.
>
> **BIO 는 학습형 모델을 위한 표기이지 만능 표기가 아닙니다.**

### 오늘 남은 시간에 BIO 를 직접 만들 일은 없습니다

BIO 를 손으로 채우는 일은 **모델을 학습시킬 때** 합니다. 토큰의 나열을 입력으로 받는 모델에게 정답도 같은 모양으로 줘야 하기 때문입니다. 우리는 모델을 학습시키지 않으므로, 오늘 남은 3절·4절에서 BIO 를 만드는 코드는 나오지 않습니다.

| 어디 | 개체를 뽑는 방법 | BIO 는 |
|---|---|---|
| **3절** 규칙 기반 | 사전을 **낱말 단위**로 맞춥니다 | 나오지 않습니다. 낱말과 그 위치를 함께 얻으므로 **(구간, 유형)** 을 바로 만듭니다 |
| **4절** 머신러닝 기반 | 학습된 모델을 부릅니다 | **모델이 만들어 내놓습니다.** 우리는 읽기만 하고, 한 줄로 다시 **(구간, 유형)** 으로 합칩니다 |
| 다음 시간 LLM | 프롬프트로 지시합니다 | 아예 없습니다. `(이름, 유형)` 을 **JSON** 으로 받습니다 |

세 방법 모두 마지막에는 **(구간, 유형)** 을 만듭니다. 우리 목적지가 그래프이고, 노드로 얹으려면 필요한 것이 개체의 **이름과 유형**이지 토큰마다의 라벨이 아니기 때문입니다.

**그럼 BIO 를 왜 배웠을까요. 읽을 줄 알아야 하기 때문입니다.** 허깅페이스 NER 모델 카드의 유형 목록, 공개 NER 데이터셋, NER 성능 지표가 전부 이 표기를 씁니다. **4-1** 에서 모델이 `B-PER` 을 돌려줄 때 그게 무슨 뜻인지 알아야 그 출력을 쓸 수 있습니다. 직접 만드는 것이 아니라 **받아 읽는 형식**으로 알아 두면 됩니다.

### ✅ 바로 확인 퀴즈

**1.** `Parkinson disease` 처럼 낱말 두 개에 걸친 개체를 BIO 로 적으면?

<details><summary>정답 보기</summary>

첫 토큰에 `B-Disease`, 이어지는 토큰에 `I-Disease` 를 붙입니다. `B` 로 시작을 표시하고 `I` 로 그 개체가 이어진다는 것을 나타내므로, 토큰 라벨만 보고도 두 토큰이 한 덩어리임을 복원할 수 있습니다.

</details>

## 2-3. 이 단원이 쓰는 개체 유형 다섯 가지

유형 목록은 정해진 하나가 아니라 **쓸 데에 맞춰 우리가 정의**합니다. 이 단원은 유형을 마음대로 고르지 않고, 앞 단원에서 적재한 **지식 그래프의 노드 레이블을 그대로** 씁니다. 여기서 뽑은 개체를 그 그래프에 얹을 것이기 때문입니다.

| 유형 | 뜻 | 예 |
|---|---|---|
| `Compound` | 약물·화합물 | omeprazole, atorvastatin, Carbidopa |
| `Gene` | 유전자 | CYP2C19, SLCO1B1, AHR |
| `Disease` | 질병 | psoriasis, Parkinson disease |
| `Symptom` | 증상 | pain, dizziness |
| `PharmacologicClass` | 약효 분류 | cannabinoids, statins |

> **유형 이름은 영어 그대로 씁니다.** 그래프의 레이블 문자열과 **글자 하나까지 같아야** 뒤 단원에서 그대로 붙기 때문입니다. 여기서 `약물` 이라고 한글로 적어 두면 나중에 그래프에 넣을 때 전부 다시 바꿔야 합니다.
>
> 날짜·기관명도 분명한 개체지만, 우리 그래프에는 그런 노드가 없어 목록에서 뺐습니다. **3-1** 에서 유전 변이 id 도 같은 이유로 뺍니다.

## 2-4. 기법의 계보: 규칙에서 LLM 까지

| 기법 | 대략 시기 | 한 줄 특징 |
|---|---|---|
| 규칙 기반 | 1990년대 | 사람이 사전과 패턴을 손으로 작성 |
| 통계 기반(CRF) | 2000년대 | 라벨 데이터로 확률을 학습, 앞뒤 문맥 반영 |
| 딥러닝(BERT) | 2018년 이후 | 문맥 임베딩으로 정확도가 크게 도약 |
| LLM | 2020년대 이후 | 재학습 없이 프롬프트로 지시해 추출 |

> **오늘은 이 표를 위에서부터 순서대로 밟습니다.** **3절**에서 가장 처음의 **규칙 기반**을 손으로 만들어 한계를 확인하고, **4절**에서 **학습된 BERT 모델**을 직접 돌려 그 한계가 넘어가는지 봅니다. 표의 마지막인 **LLM** 은 다음 시간에 씁니다.

**CRF 는 왜 건너뛰나요?** 그리고 BERT 는 **4절에서 돌려는 보지만** 우리 도메인에 맞게 **학습시키지는 않습니다.** CRF 와 BERT 계열로 새 유형을 뽑으려면 **그 도메인의 라벨 데이터**를 먼저 만들어야 하는데, 논문에 개체를 사람이 손으로 표시한 데이터가 수천 문장 필요하고, 우리에게는 그 데이터가 없습니다. **4-3 에서 보겠지만** 이미 학습된 일반 모델로는 우리 유형 5종을 낼 수 없습니다. 대신 어디서 만나는지는 알아 두세요.

BERT 계열은 **허깅페이스**(임베딩 단원에서 만난 그 허브)에서 이미 학습된 NER 모델을 받아 문장을 넣으면 바로 개체가 나옵니다. **4절**에서 직접 돌려 봅니다.

> 그 모델들이 잡는 `PER`(사람)·`ORG`(기관)·`LOC`(장소) 는 NER 자료를 열면 거의 항상 첫 줄에 나오는 **표준 유형 이름**입니다. 뜻과 유래는 **4-1** 에서 자세히 봅니다. 우리가 **2-3** 에서 유형 다섯 가지를 직접 정한 것처럼, 그 모델들도 자기 목적에 맞는 목록을 정해 둔 것뿐입니다.

---
# 3. 규칙 기반 NER 체험

계보의 첫 단계를 직접 만들어 **무엇을 잡고 어디서 막히는지** 확인합니다. 그 막히는 지점이 다음 시간 LLM 이 필요한 이유가 됩니다.

- **3-1** 토크나이저와 표준 사전(사전 축과 패턴 축)
- **3-2** 첫 시도: 사전과 낱말을 모두 소문자로 바꿔 맞추기
- **3-3** 약물 상품명에도 같은 함정
- **3-4** 고쳐 쓰기: 대소문자를 지킨다
- **3-5** 그래도 못 잡는 것들과 한계 정리

## 3-1. 토크나이저와 표준 사전

### 왜 직접 만들어 볼까요?
규칙 기반 NER 은 아이디어가 단순합니다. **아는 이름을 사전에 적어 두고, 문장에 그 이름이 나오면 유형을 붙인다**. 직접 만들어 보면 이 방식이 **무엇까지 잡고 어디서 막히는지**를 확인할 수 있습니다.

먼저 원문을 **낱말**로 쪼갭니다. 한국어라면 형태소 분석기가 필요하지만, 영어는 낱말이 이미 공백과 구두점으로 갈라져 있어 **정규식 한 줄**이면 됩니다. 다만 그 한 줄이 무엇을 낱말로 볼지 정하기 때문에, 숫자·퍼센트처럼 영문자로 시작하지 않는 것은 처음부터 빠집니다.

In [ ]:
import re

# 규칙 기반 NER 의 첫 부품 = 토크나이저. 사전과 맞춰 볼 '낱말'의 단위를 여기서 정한다.
# 한국어라면 형태소 분석기가 필요하지만, 영어는 낱말이 공백과 구두점으로 이미 갈라져 있다.
# 유전자 기호에는 하이픈과 숫자가 섞여 있으므로(HLA-A, CYP2C19) 둘 다 낱말의 일부로 본다.
# 대신 이 규칙 때문에 'Warfarin-related' 도 낱말 하나로 끊긴다. 뒤에서 이것이 경계 문제로 돌아온다.
TOKEN = re.compile(r'[A-Za-z][A-Za-z0-9\-]*')

papers = load_jsonl('core_papers.jsonl')
text = papers[0]['text']                     # 이 발췌를 이 단원 내내 데모 원문으로 쓴다
print(papers[0]['doc_id'], '/ 발췌', len(text), '자')

In [ ]:
# 이 정규식은 영문자로 시작하는 것만 잡는다. 숫자와 퍼센트는 처음부터 낱말로 보지 않는다.
print(TOKEN.findall(text)[:16])

### 규칙 기반의 두 축: 사전과 패턴

**2-4** 의 계보 표는 규칙 기반을 "사람이 **사전과 패턴**을 손으로 작성"이라고 적었습니다. 축이 둘입니다. **이름을 적어 둔 사전**으로 잡는 개체가 있고, **생김새가 정해진 패턴**으로 잡는 개체가 있습니다.

유전 변이 id 가 패턴 쪽 개체입니다. `rs` 뒤에 숫자가 붙는 모양이 고정돼 있어, 방금 만든 것과 같은 정규식 한 줄이면 잡힙니다.

In [ ]:
# 규칙 기반의 다른 축 = 패턴. 생김새가 정해진 개체는 사전 없이 정규식만으로 잡는다.
VARIANT = re.compile(r'rs\d+')                # 유전 변이 id 는 'rs' 뒤에 숫자가 붙는 모양으로 고정돼 있다
variant_paper = papers[2]                     # 이 코퍼스에서 변이 id 가 나오는 논문
print(variant_paper['doc_id'], '->', VARIANT.findall(variant_paper['text']))

> 변이 id 는 **사전에 담는다는 발상 자체가 성립하지 않습니다.** 사람 유전체에 알려진 변이만 수억 개이고 지금도 늘어납니다. 대신 생김새가 고정돼 있어 정규식이 답입니다. 날짜·용량(`50 mg`)·이메일 주소도 같은 부류입니다.
>
> 반대로 **이름**은 생김새에 규칙이 없습니다. `omeprazole` 과 `psoriasis` 와 `CYP2C19` 를 한 패턴으로 묶을 방법이 없습니다. 그래서 이름 쪽은 **사전**이 답입니다.
>
> 오늘 뽑을 유형 5종은 전부 이름 쪽이라, 이 시간 남은 부분은 **사전 축**만 다룹니다. 변이는 우리 그래프에 담을 노드가 없어 개체 목록에도 넣지 않습니다.

### ✅ 바로 확인 퀴즈

**1.** 변이 id `rs9923231` 은 사전에 없는데도 잡혔습니다. 같은 방식으로 약물 이름 `omeprazole` 을 잡을 수 없는 이유는?

<details><summary>정답 보기</summary>

변이 id 는 `rs` 뒤에 숫자가 붙는 **생김새가 고정**돼 있어 정규식 하나로 전부 잡히지만, 약물 이름은 생김새에 규칙이 없기 때문입니다. `omeprazole`·`psoriasis`·`CYP2C19` 를 한 패턴으로 묶을 방법이 없으니 이름 쪽은 **사전**을 쓸 수밖에 없습니다. 반대로 변이는 알려진 것만 수억 개라 **사전에 담는 쪽이 불가능**합니다.

</details>

그러니 이제 **사전**이 필요합니다. 손으로 이름을 적는 대신, 이 단원에서는 **실제 표준 사전**을 그대로 씁니다. 앞 단원에서 적재한 지식 그래프의 노드 이름과 그 id 를 모아 둔 파일입니다(출처는 Hetionet 과 RxNav, 이 단원의 `실습_가이드.md` 에 적어 두었습니다).

In [ ]:
# 규칙 기반 NER 의 핵심 부품 = 개체 사전(gazetteer). 손으로 적지 않고 표준 사전을 그대로 쓴다.
# 이 사전은 지식 그래프의 노드 이름과 그 id 를 담고 있어, 뒤 단원에서 개체를 그래프에 붙일 때 그대로 쓰인다.
dictionary = load_json('name2id.json')

# entries: 소문자로 통일한 이름 -> 네 칸짜리 dict. 각 칸의 뜻은 이렇다.
#   id        그래프의 노드 id. 개체를 그래프에 붙일 때 이름 대신 이 값을 쓴다 (Compound::DB00682)
#   label     유형. Compound, Disease, Symptom, PharmacologicClass 네 가지가 들어 있다
#   canonical 표준 표기. 키가 동의어면 값이 다르다 (4-hydroxybutyric acid -> Gamma Hydroxybutyric Acid)
#   source    이 항목을 가져온 곳. hetionet 아니면 rxnav
entries = dictionary['entries']

# genes: 유전자 기호 -> id. 유형이 Gene 인 항목은 entries 에 없고 여기 따로 담겨 있다.
# 이름을 소문자로 바꾸지 않고 원문 표기 그대로 둔 것이 entries 와 다른 점이다.
genes = dictionary['genes']

print('이름 사전:', len(entries), '건 / 유전자 기호:', len(genes), '건')

In [ ]:
# 이름 사전의 한 항목은 id, label, canonical, source 네 칸을 담는다. 그중 id 가 그래프의 노드를 가리킨다.
print(entries['warfarin'])

In [ ]:
# 유전자 칸은 값이 id 문자열 하나뿐이라 모양이 다르다.
print('CYP2C19 ->', genes['CYP2C19'])

> 사전 항목이 이름만이 아니라 **id 까지** 들고 있는 것이 중요합니다. `warfarin` 은 `Compound::DB00682` 라는 그래프 노드를 가리킵니다. 나중에 논문에서 뽑은 개체를 그래프에 붙일 때, 이름이 아니라 **이 id 로** 붙입니다. 이유는 이 단원 뒤에서 자세히 다룹니다.
>
> 유전자 기호가 `entries` 가 아니라 `genes` 라는 **별도 칸**에 들어 있는 데에도 이유가 있습니다. **3-2** 에서 바로 확인합니다.

## 3-2. 첫 시도: 사전과 낱말을 모두 소문자로 바꿔 맞추기

### 왜 이렇게 해 볼까요?
사전 키는 **소문자로 통일**돼 있습니다. 그러니 낱말도 소문자로 바꿔 맞추면 되겠다고 생각하기 쉽습니다. 실제로 흔히 이렇게 씁니다. 결과를 봅시다.

In [ ]:
# 흔한 첫 시도. 사전도 낱말도 소문자로 바꿔 맞춘다
gene_lower = {symbol.lower(): symbol for symbol in genes}   # 소문자 기호 -> 원래 기호. 대소문자 정보를 여기서 버린다


def naive_ner(text):
    """낱말과 사전을 모두 소문자로 바꿔 맞춘 개체 목록을 돌려준다."""
    found = []
    for match in TOKEN.finditer(text):
        word = match.group().lower()
        if word in gene_lower:               # 유전자 사전을 먼저 본다. 같은 철자가 양쪽에 있으면 유전자로 정해진다
            found.append({'name': match.group(), 'type': 'Gene'})
        elif word in entries:
            found.append({'name': match.group(), 'type': entries[word]['label']})   # 유형은 사전이 적어 둔 label 을 그대로 쓴다
    return found


naive_found = naive_ner(text)
print('소문자 매칭:', len(naive_found), '건')

In [ ]:
# 유전자로 분류된 이름만 따로 모아 본다. 이 목록에 끼어든 것이 바로 아래에서 문제가 된다.
print(sorted({ent['name'] for ent in naive_found if ent['type'] == 'Gene'}))

> 유전자로 잡힌 목록 끝에 **`large`** 가 있습니다. 논문 문장은 "a **large** proportion of the cohort" 라고 말했을 뿐인데 이게 유전자가 됐습니다. 이렇게 개체가 아닌 낱말을 개체로 잘못 잡는 것을 **오탐**(false positive)이라고 합니다. 왜 이런 일이 생기는지 확인해 봅니다.

In [ ]:
# LARGE 는 실제로 있는 유전자다. 그래서 소문자로 바꾸면 흔한 형용사 large 가 그 유전자로 잡힌다.
print('LARGE ->', genes['LARGE'])
print('사전에 large 라는 약물이나 질병 이름이 있나?', 'large' in entries)

In [ ]:
# 이런 이름이 몇 개나 되는지는 사전이 이미 세어 두었다.
print('흔한 영어 낱말과 철자가 같은 유전자 기호:', len(dictionary['gene_stopwords']), '개')

In [ ]:
# 목록을 직접 본다. 전부 실재하는 유전자 기호이면서 동시에 평범한 영어 낱말이다.
print(dictionary['gene_stopwords'])

> **유전자 기호는 대소문자가 곧 뜻입니다.** `LARGE`·`CAT`·`ACT`·`CLOCK`·`SET`·`IMPACT` 는 전부 실재하는 유전자 기호이고, 동시에 전부 평범한 영어 낱말입니다. 소문자로 바꿔 맞추는 순간 이 둘이 구분되지 않습니다.
>
> 그래서 사전이 유전자 기호를 `entries` 와 **섞지 않고** `genes` 칸에 따로 담아 둔 것입니다. 매칭 방식을 유형별로 달리 하라는 뜻입니다.

## 3-3. 약물 상품명에도 같은 함정

유전자만의 문제로 보이지만 아닙니다. 약에는 **상품명**이 있고, 제약사는 기억하기 쉬운 이름을 붙입니다. 그러다 보니 평범한 영어 낱말이 그대로 약 이름이 된 것들이 있습니다.

In [ ]:
# 같은 함정이 약물 이름에도 있다. 사전이 이쪽도 따로 표시해 두었다.
brand_stopwords = set(dictionary['brand_stopwords'])   # 낱말마다 있는지 물어볼 것이라 집합으로 둔다
print('평범한 영어 낱말과 철자가 같은 상품명:', len(brand_stopwords), '개')

In [ ]:
# 어떤 낱말이 어떤 약으로 둔갑하는지 짝지어 본다(왼쪽이 평범한 낱말, 오른쪽이 그 상품명의 실제 약).
for name in dictionary['brand_stopwords']:
    print(f"  {name:12s} -> {entries[name]['canonical']} ({entries[name]['label']})")

In [ ]:
# 지금 이 발췌에서 실제로 그런 낱말이 쓰였는지 센다.
print('이 발췌에서:', [w for w in TOKEN.findall(text) if w.lower() in brand_stopwords])

> `perform` 이 멘톨이 되고 `today` 가 항생제가 됩니다. 실제로 그런 이름의 제품이 있어서 사전에 들어간 것이지, 사전이 틀린 것이 아닙니다. **논문 문장의 `perform` 은 그냥 동사인데 사전은 그걸 구분할 수 없습니다.**
>
> 다만 이 발췌에서는 그런 낱말이 **하나도 나오지 않았습니다**(위 출력이 빈 목록입니다). 지금 당장 손해를 보고 있지는 않다는 뜻입니다. 그래도 규칙에는 넣어 둡니다. 논문을 수백 편 처리하면 반드시 만나고, 그때는 이미 그래프에 잘못 들어간 뒤이기 때문입니다.
>
> 정리하면 함정은 **사전에 적힌 이름이 평범한 낱말과 철자가 같을 때** 늘 생기는 문제입니다. 유전자 기호 33개, 약물 상품명 8개가 지금 사전에 표시돼 있습니다.

## 3-4. 고쳐 쓰기: 대소문자를 지킨다

### 왜 이렇게 고칠까요?
두 함정은 고치는 방법이 같습니다. **원문에 적힌 대소문자를 그대로 지켜서 맞추는 것**입니다.

- 유전자 기호는 아예 대소문자까지 맞춰 찾습니다(`LARGE` 는 잡고 `large` 는 안 잡습니다).
- 나머지 이름은 소문자로 맞춰도 됩니다(`Omeprazole` 이든 `omeprazole` 이든 같은 약입니다). 다만 **상품명 8개만은 예외로**, 소문자로 쓰였으면 평범한 낱말로 봅니다.

In [ ]:
def rule_based_ner(text):
    """사전에 있는 낱말을 (구간, 유형)이 붙은 개체 dict 리스트로 돌려준다."""
    found = []
    for match in TOKEN.finditer(text):       # finditer 는 낱말과 함께 그 위치까지 준다
        word = match.group()                 # 소문자로 바꾸지 않고 원문 표기를 그대로 든다
        if word in genes:                    # 유전자 기호는 대소문자가 곧 뜻이라 그대로 맞춘다
            entity_type = 'Gene'
        elif word.lower() in entries:        # 나머지 이름은 소문자로 맞춘다
            if word.lower() in brand_stopwords and word.islower():
                continue                     # 단 상품명이 소문자로 쓰였으면 평범한 낱말이다
            entity_type = entries[word.lower()]['label']
        else:
            continue                         # 사전에 없는 낱말은 여기서 통째로 빠진다(3-5 에서 다시 본다)
        found.append({'name': word, 'type': entity_type,
                      'start': match.start(), 'end': match.end()})   # 2-1 에서 만든 구간을 여기서도 붙인다
    return found


careful_found = rule_based_ner(text)
print('대소문자를 지킨 매칭:', len(careful_found), '건')

In [ ]:
# 두 결과를 집합으로 빼면 대소문자를 지켜서 정확히 무엇이 걸러졌는지 남는다.
print('사라진 오탐:', {e['name'] for e in naive_found} - {e['name'] for e in careful_found})

In [ ]:
# 각 개체가 2-1 과 같은 모양의 (구간, 유형) 쌍으로 나온다. start 와 end 로 원문을 되짚을 수 있다.
for ent in careful_found[:6]:                # 28건 전부는 길어서 앞 6건만 본다
    print(ent, '->', text[ent['start']:ent['end']])

> 오탐 하나가 사라졌고, 각 개체는 2-1 과 같은 **(구간, 유형)** 쌍으로 나옵니다. 이 발췌에서는 한 건이지만, 사전에 그런 이름이 33개 있으니 논문을 수백 편 처리하면 그만큼 늘어납니다.

### ✅ 바로 확인 퀴즈 (3-2·3-3·3-4)

**1.** 유전자 기호를 소문자로 바꿔 매칭하면 안 되는 이유는?

<details><summary>정답 보기</summary>

`LARGE`·`CAT`·`SET` 처럼 **평범한 영어 낱말과 철자가 같은 유전자 기호**가 실재하기 때문입니다. 소문자로 바꾸면 본문의 형용사 `large` 가 유전자 `LARGE` 로 잡힙니다. 유전자 기호는 대소문자가 곧 뜻이라 원문 표기 그대로 맞춰야 합니다.

</details>

**2.** 그런데 상품명은 왜 유전자 기호와 **다르게** 다루나요? (`brand_stopwords` 에 든 이름만 소문자일 때 걸러 냅니다)

<details><summary>정답 보기</summary>

약물 이름은 대소문자가 뜻을 바꾸지 않기 때문입니다. `Omeprazole` 이든 `omeprazole` 이든 같은 약이라 소문자로 맞춰야 둘 다 잡힙니다. 그래서 유전자처럼 통째로 대소문자를 지킬 수는 없고, **평범한 낱말과 철자가 같은 상품명 8개만** 예외로 두어 소문자로 쓰였을 때 넘깁니다.

</details>

## 3-5. 그래도 못 잡는 것들과 한계 정리

이제 반대쪽을 봅니다. **원문에 분명히 있는데 규칙 기반이 놓친 것**은 무엇일까요.

In [ ]:
# 발췌에 분명히 나오는데 사전 매칭이 못 잡는 이름들을 직접 확인해 본다.
for name in ['statins', 'proton pump inhibitors', 'Warfarin']:
    in_dictionary = name.lower() in entries or name in genes   # 사전 두 칸 중 어느 쪽에라도 있으면 True
    caught = any(ent['name'] == name for ent in careful_found)   # 규칙 기반이 실제로 잡았는지
    print(f'{name:24s} 원문에 있나 {name in text} / 사전에 있나 {in_dictionary} / 잡았나 {caught}')

In [ ]:
# Warfarin 은 사전에 있는데도 못 잡았다. 원문에 어떻게 적혀 있는지 보면 이유가 보인다.
start = text.find('Warfarin')
print(repr(text[start:start + 16]))

> 세 가지가 각각 다른 이유로 빠졌습니다.
>
> - `statins`·`proton pump inhibitors`: **사전에 없습니다.** 사전은 2016년에 정리된 그래프에서 왔고, 논문은 최신 표현을 씁니다. 이것이 **미등록어** 문제입니다.
> - `Warfarin`: 사전에 **있는데도** 못 잡았습니다. 원문이 `Warfarin-related` 라 낱말이 통째로 하나로 끊겼기 때문입니다. 이것이 **경계** 문제입니다.

In [ ]:
# 낱말 하나 단위로 맞추는 한 이런 이름은 구조적으로 못 잡는다.
multiword = [name for name in entries if ' ' in name]   # 사전 키에 공백이 들었다면 낱말 여러 개짜리 이름이다
print('이름에 공백이 든 항목:', len(multiword), '건 /', len(entries), '건')

In [ ]:
# 공백이 든 이름 다섯 개. 낱말 하나로는 못 맞춘다
print(multiword[:5])

> 낱말 하나 단위로 맞추는 방식은 **공백이 든 이름**을 구조적으로 못 잡습니다. 사전 항목의 38% 가 그렇습니다. 낱말을 두 개·세 개씩 묶어 가며 맞추도록 고칠 수는 있지만, 그럴수록 규칙이 늘어납니다.

In [ ]:
# 대소문자를 지켜도 오탐이 전부 사라지지는 않는다. 같은 코퍼스의 다른 논문을 보자.
sleep_paper = papers[5]                      # 수면 장애 논문. 약어가 많아 충돌이 잘 드러난다
print(sleep_paper['doc_id'], sleep_paper['title'][:60])

In [ ]:
# 이 논문에서 유전자로 잡힌 것만 본다. 대소문자를 지켰는데도 남는 오탐이 있다.
print([ent['name'] for ent in rule_based_ner(sleep_paper['text']) if ent['type'] == 'Gene'])

> 대소문자를 지켜도 남는 오탐이 있습니다. 이 논문에서 `PSD` 는 수술 후 수면 장애(postoperative sleep disturbance), `OTC` 는 일반의약품(over the counter)의 약자인데, 둘 다 같은 철자의 **유전자 기호가 실재**합니다. 사전 매칭은 문맥을 읽지 못하므로 이런 **약어 충돌**을 가릴 수 없습니다.

### 한계 정리

| 한계 | 이 발췌에서 본 것 | 왜 |
|---|---|---|
| 미등록어 | `statins`, `proton pump inhibitors` | 사전에 없는 이름은 못 잡는다 |
| 경계 | `Warfarin-related` 안의 `Warfarin`, 공백이 든 이름 | 낱말을 어디서 끊을지 규칙이 정한다 |
| 대소문자 | `large` 가 유전자로 | 소문자로 바꾸면 뜻이 다른 이름이 겹친다. 유전자 기호 33개·약물 상품명 8개가 그렇다 |
| 약어 충돌 | `PSD`, `OTC` | 문맥을 못 읽어 같은 철자를 가릴 수 없다 |

> 새 이름이 나올 때마다 사전을 채우고, 예외가 나올 때마다 규칙을 덧대는 일은 **끝이 없습니다**. 그래서 계보의 다음 칸으로 넘어갑니다. **4절에서 학습된 NER 모델**을 직접 돌려, 사전 없이 문맥으로 잡으면 이 벽이 사라지는지 확인합니다.
>
> 다만 사전을 버리는 것은 아닙니다. LLM 이 뽑아 온 이름을 **그래프의 어느 노드에 붙일지** 정하려면 결국 이 사전이 필요합니다. 그 이야기는 이 단원 뒤에서 이어집니다.

### 🖐️ 함께 따라하기: 사내 기술 문서에서 오탐 찾아내기

**같은 함정이 의료 밖에서도 똑같이 생깁니다.** `Go`·`Rust`·`Swift`·`Spring`·`Spark` 는 전부 실재하는 기술명이면서 동시에 평범한 영어 낱말입니다. 유전자 기호 `LARGE`·`SET` 과 구조가 같습니다.

아래 제공 셀을 실행한 뒤, 기술 문서 한 토막(`doc`)에 **소문자로 바꿔 맞추기**와 **대소문자를 지킨 맞추기**를 각각 적용해 무엇이 걸러지는지 보세요.

In [ ]:
# [제공 코드] 사내 기술 문서용 작은 사전. 실행만 하세요.
# 구조는 name2id.json 과 같다. 소문자로 통일한 이름 -> 유형.
tech_names = {'go': 'Language', 'rust': 'Language', 'swift': 'Language', 'python': 'Language',
              'spring': 'Framework', 'react': 'Framework', 'django': 'Framework',
              'kafka': 'Tool', 'airflow': 'Tool', 'spark': 'Tool'}

# 평범한 영어 낱말과 철자가 같은 기술명. 유전자 기호 33개와 똑같은 함정이다.
tech_stopwords = {'go', 'rust', 'swift', 'spring', 'spark'}

doc = ('The team will go over the migration plan next spring. '
       'We run Airflow for scheduling and Kafka for the event log, and the API is written in Go.')
print(doc)

- `TOKEN.findall(doc)` 로 낱말을 뽑고, 소문자로 바꾼 낱말이 `tech_names` 에 있으면 기술명 후보입니다.
- 그중 **소문자로 쓰였고**(`word.islower()`) `tech_stopwords` 에 든 것은 평범한 낱말로 보고 건너뜁니다. `rule_based_ner` 의 상품명 처리와 같은 규칙입니다.
- **확인 기준**: 걸러진 오탐이 `go`·`spring` **2건**이면 맞은 것입니다. 문장 끝의 대문자 `Go` 는 **걸러지지 않고 남습니다**. 같은 철자를 대소문자가 갈라 준 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 빈 집합 noisy, clean 을 만들고 TOKEN.findall(doc) 의 낱말을 돈다
# 2) 소문자로 바꾼 낱말이 tech_names 에 없으면 건너뛰고, 있으면 원래 낱말을 noisy 에 담는다
# 3) 소문자로 쓰였고 tech_stopwords 에 든 낱말은 건너뛰고, 나머지만 clean 에 담는다
# 4) noisy, clean, 그리고 걸러진 오탐(noisy - clean)을 각각 출력한다

### ✅ 바로 확인 퀴즈

**1.** 규칙 기반 NER 이 `statins` 를 못 잡은 것과 `Warfarin-related` 안의 `Warfarin` 을 못 잡은 것은 서로 다른 문제입니다. 각각 무엇인가요?

<details><summary>정답 보기</summary>

`statins` 는 **미등록어**입니다. 사전 자체에 그 이름이 없습니다. `Warfarin` 은 사전에 있지만 원문에서 `Warfarin-related` 로 붙어 있어 낱말이 통째로 하나로 끊겼습니다. 이건 **경계** 문제이고, 사전을 아무리 키워도 해결되지 않습니다.

</details>

---
# 4. 머신러닝 기반 NER 체험

계보의 다음 단계입니다. 규칙 기반은 **사람이 사전과 패턴을 적어** 개체를 잡았고, 3절에서 그 방식이 막히는 자리를 넷 봤습니다. 이번에는 **라벨 데이터로 학습된 모델**이 그 벽을 넘는지 직접 확인합니다.

- **4-1** 학습된 NER 모델 돌려 보기
- **4-2** `pipeline` 의 입력과 출력
- **4-3** 우리 논문에 그대로 쓰면 어떻게 되나
- **4-4** 허브에서 모델 고르기

## 4-1. 학습된 NER 모델 돌려 보기

**2-2** 에서는 BIO 태그를 우리 손으로 붙였습니다. 학습된 모델도 정말 그 형식으로 답하는지 확인합니다. 허깅페이스에서 **이미 학습된 NER 모델**(`dslim/bert-base-NER`)을 받아 문장 하나를 넣어 보겠습니다. 계보 표(**2-4**)의 **딥러닝(BERT)** 칸에 있는 그 모델입니다.

> 처음 실행할 때 모델을 **약 430MB** 내려받습니다. 잠시 걸리고, 그 뒤로는 캐시에서 바로 뜹니다. `HF_TOKEN` 이 없다는 안내 한 줄이 뜰 수 있는데 무시해도 됩니다.

### 먼저: 이 모델이 붙이는 유형 네 가지

출력을 읽으려면 라벨의 뜻부터 알아야 합니다. 이 모델의 유형은 우리 5종(`Compound`·`Gene`·`Disease`·`Symptom`·`PharmacologicClass`)과 **완전히 다른 목록**이고, 넷뿐입니다. 넷 다 영어 낱말을 줄인 것입니다.

| 유형 | 무엇의 약자 | 뜻 | 이 절의 예 |
|---|---|---|---|
| `PER` | **Per**son | 사람 이름 | `Sarah Kim` |
| `ORG` | **Org**anization | 회사·기관·단체 | `Neo4j` |
| `LOC` | **Loc**ation | 장소·지명 | `London` |
| `MISC` | **Misc**ellaneous | 위 셋에 안 드는 고유명사(국적·사건명·제품명 등) | 아래 출력엔 없고 **4-3** 에서 만납니다 |

이 목록은 이 모델이 학습한 **CoNLL-2003** 데이터가 정해 둔 것입니다(뉴스 기사 데이터셋이고, **4-4** 에서 모델 카드로 확인합니다). 그래서 뉴스에 흔한 사람·기관·장소에 강하고, 약물이나 유전자는 애초에 목록에 없습니다.

앞에 붙는 `B-`·`I-` 는 **2-2** 에서 본 그것입니다. 즉 `B-PER` 은 "사람 이름이 여기서 시작", `I-PER` 은 "그 사람 이름이 이어짐" 입니다.

In [ ]:
# 학습된 NER 모델. 처음 실행하면 약 430MB 를 내려받는다
from transformers import logging as hf_logging, pipeline

hf_logging.set_verbosity_error()     # 모델을 올릴 때 뜨는 안내 표를 끈다
hf_logging.disable_progress_bar()    # 가중치를 올리는 진행 막대를 끈다

ner = pipeline('token-classification', model='dslim/bert-base-NER')
news = 'Sarah Kim joined Neo4j in London last spring.'

# 출력은 dict 의 리스트. entity 칸이 BIO 라벨이다
print(ner(news))

> **모델이 `B-PER`·`I-PER`·`B-ORG` 를 그대로 내놓습니다.** `Sarah`·`Kim` 두 토큰이 사람 이름 하나(`B-PER`+`I-PER`), `London` 이 장소 하나(`B-LOC`)로 붙었습니다. 우리가 손으로 적은 그 표기입니다. BIO 는 교재용 연습이 아니라 **토큰 분류형 NER 모델이 실제로 쓰는 출력 형식**입니다. 학습된 모델을 받아 쓰거나 직접 학습시킬 때 이 라벨을 그대로 만납니다.
>
> **2-2 에서 말한 그 자리입니다.** `Neo4j` 가 `Neo`·`##4`·`##j` **세 토큰으로 쪼개져** `B-ORG`·`I-ORG`·`I-ORG` 를 달았습니다. 공백이 하나도 없는 낱말인데 `I-` 가 붙었습니다. 모델이 낱말을 우리처럼 `split()` 으로 끊지 않고 더 잘게 나눠 쓰기 때문입니다(`##` 는 앞 토큰에 붙는다는 표시입니다). 토큰을 어떻게 끊든 **`B-` 로 시작하고 `I-` 로 잇는 규칙은 같습니다.**

그런데 우리가 2-1 에서 만든 것은 토큰 라벨이 아니라 **(구간, 유형)** 쌍이었습니다. 같은 모델에 `aggregation_strategy` 한 줄만 더 주면 B-/I- 를 합쳐 그 모양으로 돌려줍니다.

In [ ]:
# aggregation_strategy 는 B- 와 I- 를 합쳐 개체 단위로 돌려준다
merged = pipeline('token-classification', model='dslim/bert-base-NER',
                  aggregation_strategy='simple')

merged_ents = merged(news)

# 합친 쪽은 entity 대신 entity_group 이 오고 start 와 end 가 붙는다
print(merged_ents)

> `Sarah`·`Kim` 이 `Sarah Kim` 하나로, 쪼개졌던 `Neo`·`##4`·`##j` 가 `Neo4j` 하나로 합쳐졌습니다. 라벨 칸의 이름이 `entity` 에서 **`entity_group`** 으로 바뀌고 `B-`·`I-` 가 사라졌으며, **`start`·`end` 가 붙어 나옵니다.**

그 `start`·`end` 로 원문을 잘라내 봅니다.

In [ ]:
# start 와 end 로 원문을 자르면 word 와 같다. 2-1 의 (구간, 유형) 쌍과 같은 것
print([news[ent['start']:ent['end']] for ent in merged_ents])

> 개체 이름과 똑같이 잘려 나옵니다. **2-1 에서 `find` 로 손수 만든 그 (구간, 유형) 쌍을 모델이 돌려준 것**입니다.
>
> 정리하면 **BIO 는 모델이 학습하고 내놓는 안쪽 표기**이고, **(구간, 유형)** 은 그것을 합쳐 우리가 쓰는 바깥쪽 형식입니다. 실무에서 학습형 NER 을 쓸 때는 이 합치는 단계까지 해야 결과를 쓸 수 있습니다.

## 4-2. `pipeline` 의 입력과 출력

방금 두 번 부른 것이 무엇을 받고 무엇을 주는지 정리합니다.

- **입력**: 문자열 하나입니다. 문장이든 문단이든 됩니다. 리스트를 주면 여러 개를 한 번에 처리합니다.
- **출력**: **dict 의 리스트**입니다. 개체를 하나도 못 찾으면 빈 리스트가 옵니다.

출력 dict 에 어떤 칸이 들었는지는 `aggregation_strategy` 를 줬는지에 따라 갈립니다.

| 칸 | 안 줬을 때 (토큰 하나가 원소) | 줬을 때 (개체 하나가 원소) |
|---|---|---|
| 라벨 | `entity` : `B-PER` 처럼 **BIO 라벨** | `entity_group` : `PER` 처럼 **유형만** |
| 이름 | `word` : 토큰 하나(`##4` 같은 조각도 그대로) | `word` : 합쳐진 개체 이름 |
| 위치 | `start` · `end` | `start` · `end` |
| 확신 | `score` : 0~1 | `score` : 합친 토큰들의 평균 |
| 순서 | `index` : 토큰 번호 | 없음 |

> `score` 는 **모델이 얼마나 확신하는지**입니다. 낮은 것을 임계값으로 버리는 것이 학습형 NER 을 쓸 때의 흔한 첫 필터입니다. 이 값은 모델이 계산한 **확률**입니다. 다음 시간에 LLM 에게도 확신도를 물어보는데, 그건 확률이 아니라 **모델의 자기 보고**라서 성격이 다릅니다(교안_02 2-1).

## 4-3. 우리 논문에 그대로 쓰면 어떻게 되나

여기까지 보면 답이 나온 것 같습니다. 3절의 규칙 기반은 사전에 없는 이름을 못 잡았는데, 이 모델은 사전 없이 문맥으로 잡습니다. 그러면 **미등록어와 경계 문제는 끝난 것 아닌가요.** 같은 모델에 **우리 논문 문장**을 넣어 봅니다.

In [ ]:
# 4-1 의 모델에 우리 논문 문장을 넣어 본다
paper_sentence = 'Carbidopa, used to treat Parkinson disease, was reported to activate AHR.'   # 2-1 의 문장. 3절에서 text 가 발췌 전체로 바뀌어 다시 담는다

for ent in merged(paper_sentence):
    print(f"{ent['word']:>12}  {ent['entity_group']:<5} {ent['score']:.2f}")

> 셋 다 틀렸습니다. 약물 `Carbidopa` 는 **기타(MISC)** 로, 그것도 `Carbidop` 까지만 잘려 나왔고, 질병 `Parkinson disease` 는 **사람 이름(PER)** 이 됐습니다. 유전자 `AHR` 은 아예 잡히지 않았습니다.

모델이 틀렸다기보다 **이 모델이 낼 수 있는 유형 자체가 다릅니다.** 직접 물어보면 알 수 있습니다.

In [ ]:
# 이 모델이 낼 수 있는 유형 종류
print(sorted({label[2:] for label in ner.model.config.id2label.values() if label != 'O'}))

> 이 모델이 낼 수 있는 유형은 `PER`·`ORG`·`LOC`·`MISC` **넷뿐**입니다. 우리가 **2-3** 에서 정한 `Compound`·`Gene`·`Disease`·`Symptom`·`PharmacologicClass` 는 **선택지에 아예 없습니다.** 유형 목록은 학습할 때 정해지고, 쓰는 쪽에서는 바꿀 수 없습니다.
>
> 정리하면 학습된 모델은 규칙 기반의 네 한계 중 **미등록어와 경계는 해결합니다.** 사전에 없는 이름도 문맥으로 잡기 때문입니다. 대신 **유형 목록을 우리가 정할 수 없습니다.** 목록을 바꾸려면 그 도메인의 라벨 데이터로 **다시 학습**시켜야 합니다. 논문에 개체를 사람이 손으로 표시한 데이터가 수천 문장 필요합니다(**2-4**).
>
> 다음 시간에 쓸 **LLM** 은 재학습 없이 **프롬프트에 유형 목록을 적어** 바꿉니다.

## 4-4. 허브에서 모델 고르기

지금까지는 `dslim/bert-base-NER` 이라는 이름을 **교재가 주었습니다.** 실무에서는 그 이름을 여러분이 찾아야 합니다. 찾는 법과 고르는 법을 순서대로 봅니다.

### 1) 후보 찾기

허브에서 NER 모델을 찾는 길은 둘입니다.

- **웹으로**: `huggingface.co/models` 에서 왼쪽 **Tasks** 필터를 **`Token Classification`** 으로 좁힙니다. NER 은 허브에서 이 이름으로 분류돼 있어서, `ner` 로만 검색하면 엉뚱한 것이 섞입니다. 여기에 언어 필터와 검색어를 더하고 정렬을 **Most downloads** 로 둡니다.
- **코드로**: 같은 일을 `huggingface_hub` 으로 합니다. 후보를 표로 뽑아 두고 비교하기에 편합니다.

In [ ]:
# NER 은 허브에서 'token-classification' 태스크로 분류돼 있다
from huggingface_hub import list_models

for model in list_models(pipeline_tag='token-classification', sort='downloads', limit=5):
    # downloads 는 최근 한 달 내려받은 횟수다. 많이 쓰인다고 우리에게 맞는 것은 아니다
    print(f'{model.id:52s} {model.downloads:>10,}  likes {model.likes}')

> 많이 쓰이는 순서로 나왔습니다. 다만 **다운로드 수는 인기이지 적합성이 아닙니다.** 1위가 뉴스 기사로 학습한 영어 모델이어도 우리 논문에 맞는다는 뜻은 아닙니다. 순위는 실행하는 시점마다 조금씩 다릅니다.

검색어를 넣어 우리 쪽으로 좁혀 봅니다.

In [ ]:
# 검색어로 좁히기
for model in list_models(pipeline_tag='token-classification', search='biomedical',
                         sort='downloads', limit=5):
    print(f'{model.id:52s} {model.downloads:>10,}')

> **4절 내내 쓴 의료 모델이 이 검색으로 맨 위에 나옵니다.** 교재가 그냥 고른 이름이 아니라 이렇게 찾은 것입니다.

### 2) 모델 카드 읽기

후보가 몇 개로 좁혀지면 **모델 카드**(허브의 그 모델 페이지)를 읽습니다. 넷만 보면 됩니다.

| 볼 것 | 왜 |
|---|---|
| **유형 목록** | 우리가 원하는 유형이 없으면 그 모델은 거기서 끝입니다. 아래 3)에서 코드로 확인합니다 |
| **학습 데이터** | 어떤 글로 배웠는지가 어디에 잘 듣는지를 정합니다. `conll2003` 은 **뉴스 기사**입니다 |
| **라이선스** | 회사 일에 쓰려면 반드시 봅니다. `mit`·`apache-2.0` 은 상업적 사용이 열려 있습니다 |
| **마지막 갱신일** | 관리되는 모델인지 가늠합니다 |

앞의 셋은 카드 위쪽 태그에 붙어 있어 코드로도 읽힙니다.

In [ ]:
# 모델 카드의 메타 정보. 태그에 라이선스와 학습 데이터가 들어 있다
from huggingface_hub import model_info

for name in ['dslim/bert-base-NER', 'd4data/biomedical-ner-all']:
    info = model_info(name)
    licenses = [tag for tag in info.tags if tag.startswith('license:')]
    datasets = [tag for tag in info.tags if tag.startswith('dataset:')]
    print(f'{name:30s}', licenses, datasets, '갱신', str(info.lastModified)[:10])

> `dslim/bert-base-NER` 이 `conll2003` 으로 배웠다는 것이 여기서 확인됩니다. **뉴스 기사 데이터셋**입니다. 그래서 사람·기관·장소는 잘 잡고 약물·유전자는 못 잡습니다. 4-3 의 결과가 이 한 줄로 설명됩니다.
>
> 의료 모델 쪽은 학습 데이터 칸이 **비어서** 나옵니다. 태그를 안 채워 둔 것이고, 흔한 일입니다. 그럴 때는 카드 본문을 직접 읽어야 합니다. **태그가 없다고 학습 데이터가 없는 것이 아니라, 적어 두지 않은 것**입니다.

### 3) 후보 비교하기

이렇게 찾은 것 중 성격이 다른 넷을 놓고 봅니다.

| 모델 | 언어·도메인 | 내놓는 유형 |
|---|---|---|
| `dslim/bert-base-NER` | 영어 일반 | 4종: `PER`·`ORG`·`LOC`·`MISC` |
| `Davlan/xlm-roberta-base-ner-hrl` | 다국어 일반 | 4종: `PER`·`ORG`·`LOC`·`DATE` |
| `Leo97/KoELECTRA-small-v3-modu-ner` | **한국어** 일반 | 15종: `PS`(인물)·`LC`(지역)·`OG`(기관)·`DT`(날짜)·`QT`(수량) 등 |
| `d4data/biomedical-ner-all` | 영어 **의료** | 43종: `Disease_disorder`·`Medication`·`Sign_symptom` 등 |

**모델을 고를 때 첫 질문은 언제나 "이 모델이 내가 원하는 유형을 내는가" 입니다.** 모델 카드에 적혀 있지만, 코드로도 확인할 수 있습니다. 가중치를 통째로 받기 전에 **설정 파일만** 내려받으면 됩니다. 모델당 수 KB 라 금방입니다.

In [ ]:
# 가중치 없이 설정 파일만 받아 유형 확인. 모델당 수 KB
from transformers import AutoConfig

for name in ['Davlan/xlm-roberta-base-ner-hrl',
             'Leo97/KoELECTRA-small-v3-modu-ner',
             'd4data/biomedical-ner-all']:
    config = AutoConfig.from_pretrained(name)      # 설정만 받는다. 모델 본체는 안 받는다
    # id2label 은 '숫자 -> BIO 라벨' 표다. 앞의 B-/I- 를 떼면 유형 이름만 남는다
    types = sorted({label.split('-', 1)[-1] for label in config.id2label.values() if label != 'O'})
    print(f'{name:40s} {len(types):>3}종  {types}')

> 이것이 **모델을 쓰기 전에 살펴보는 방법**입니다. `config.id2label` 은 모델이 낼 수 있는 라벨의 전체 목록이고, 여기에 없는 유형은 그 모델에서 **절대 나오지 않습니다.** 430MB 를 받고 나서 "내가 원하던 유형이 아니네" 하는 일을 막아 줍니다.
>
> 의료 모델은 43종을 한 줄에 쏟아 냅니다. 눈으로 훑기보다 **우리 5종이 그 안에 있는지** 코드로 묻는 편이 확실합니다.
>
> 목록 안에 `Non[biological](Detailed_description` 처럼 **깨진 이름**이 섞여 있는 것도 보입니다. 모델을 올린 사람이 라벨 표를 잘못 적어 둔 것이고, 공개 모델에서 드물지 않습니다. 카드에 적힌 설명과 실제 `id2label` 이 다를 수 있으니 **코드로 확인하는 습관**이 필요한 이유이기도 합니다.

In [ ]:
# 의료 모델 43종에 우리 유형이 있는지
bio_config = AutoConfig.from_pretrained('d4data/biomedical-ner-all')
bio_types = sorted({label.split('-', 1)[-1] for label in bio_config.id2label.values() if label != 'O'})

print('질병, 약물, 증상 쪽:', [t for t in bio_types if t in ('Disease_disorder', 'Medication', 'Sign_symptom')])
# 나머지 둘은 이름에 Gene 이나 Pharmac 가 들어갈 것이다
print('유전자와 약효분류 쪽:', [t for t in bio_types if 'Gene' in t or 'Pharmac' in t])

> 의료 모델에는 `Disease_disorder`·`Medication`·`Sign_symptom` 이 있어 우리 `Disease`·`Compound`·`Symptom` 과 가깝습니다. 그런데 **`Gene` 과 `PharmacologicClass` 쪽은 빈 목록입니다.** 43종을 갖춘 의료 전용 모델인데도 우리가 쓸 유형이 다 있지는 않습니다. 도메인을 맞춰도 유형 목록이 정확히 겹치지는 않는다는 뜻이고, 뽑은 개체를 **우리 그래프에 그대로 얹어야** 하는 우리에게는 이 차이가 그대로 문제가 됩니다.
>
> 그래서 다음 시간에는 **유형 목록을 우리가 프롬프트에 적어 주는** LLM 으로 갑니다. 학습된 모델을 고르는 것이 아니라, 우리 목록을 모델에게 알려 주는 쪽으로 방향이 바뀝니다.

---
## 이번 강의 정리

| 주제 | 핵심 |
|---|---|
| IE 4단계 | 개체 인식 → 스키마 설계 → 관계 추출 → 정규화 (오늘은 개체 인식) |
| NER | 개체의 **구간(span)과 유형**을 찾기. 학습형 모델의 표기는 **BIO 태깅** |
| BIO 의 `B-` | 같은 유형 개체가 나란히 붙은 **경계**를 가른다 |
| 유형 5종 | 그래프 노드 레이블과 같은 문자열: `Compound`·`Gene`·`Disease`·`Symptom`·`PharmacologicClass` |
| 일반 NER 모델 | `PER`·`ORG`·`LOC`·`MISC` 로 **유형이 고정**돼 우리 5종을 못 낸다 |
| 규칙 기반 NER (3절) | 표준 사전 매칭. 미등록어·경계·대소문자·약어 충돌에서 막힘 |
| 머신러닝 기반 NER (4절) | 사전 없이 문맥으로 잡지만, **유형 목록이 학습할 때 고정**된다 |

- 우리 그래프는 2016년 데이터라 새 논문의 사실이 없습니다. 그래서 논문에서 개체·관계를 뽑아 **그래프에 얹는** 일을 시작했습니다.
- 그 첫 단계가 **NER** 입니다. `find` 로 **구간**을 만들고, 그것을 **BIO 태깅**으로 옮겨 적어 봤습니다.
- 그다음 계보를 순서대로 밟았습니다. **규칙 기반**(3절)은 빠르고 사전의 id 까지 바로 얻지만 네 지점에서 막혔고, **학습된 모델**(4절)은 그중 미등록어·경계는 넘었지만 **우리가 원하는 유형을 낼 수 없었습니다.**
- 남은 방법이 **LLM** 입니다. **유형 목록을 우리가 정하면서도 문맥을 읽는 방법**이 다음 시간의 주제입니다.

## ⏭️ 예고: 다음 시간

규칙 기반이 막혔던 바로 그 발췌를, 이번엔 **LLM** 에게 시켜 봅니다. 재학습도 사전도 없이 **프롬프트에 유형 목록을 적어** `statins`·`Warfarin` 까지 어떻게 잡는지, 그리고 그 결과를 프로그램이 바로 쓰도록 **JSON** 으로 받는 법까지 배웁니다. 그러면서 새 문제도 하나 만납니다. LLM 이 **사전에 없는 이름**을 뽑아 오면 그것을 그래프에 어떻게 넣어야 할까요.

수고하셨습니다!